In [8]:
from pathlib import Path

txt_path = Path("../data/MACCROBAT2018/15939911.txt")

text = txt_path.read_text(encoding="utf-8")

print(text)

CASE: A 28-year-old previously healthy man presented with a 6-week history of palpitations.
The symptoms occurred during rest, 2–3 times per week, lasted up to 30 minutes at a time and were associated with dyspnea.
Except for a grade 2/6 holosystolic tricuspid regurgitation murmur (best heard at the left sternal border with inspiratory accentuation), physical examination yielded unremarkable findings.
An electrocardiogram (ECG) revealed normal sinus rhythm and a Wolff– Parkinson– White pre-excitation pattern (Fig.1: Top), produced by a right-sided accessory pathway.
Transthoracic echocardiography demonstrated the presence of Ebstein's anomaly of the tricuspid valve, with apical displacement of the valve and formation of an “atrialized” right ventricle (a functional unit between the right atrium and the inlet [inflow] portion of the right ventricle) (Fig.2).
The anterior tricuspid valve leaflet was elongated (Fig.2C, arrow), whereas the septal leaflet was rudimentary (Fig.2C, arrowhead)

In [9]:
ann_path = Path("../data/MACCROBAT2018/15939911.ann")

annotations =ann_path.read_text(encoding="utf-8")

print(annotations[:2000])

T1	Age 8 19	28-year-old
T2	History 20 38	previously healthy
T3	Sex 39 42	man
T4	Clinical_event 43 52	presented
E1	Clinical_event:T4 
T5	Sign_symptom 31 38	healthy
E2	Sign_symptom:T5 
T6	Duration 60 66	6-week
E3	Duration:T6 
T7	Sign_symptom 78 90	palpitations
E4	Sign_symptom:T7 
T8	Coreference 96 104	symptoms
E5	Coreference:T8 
R1	IDENTICAL Arg1:E5 Arg2:E4	
T9	Clinical_event 121 125	rest
E6	Clinical_event:T9 
R2	MODIFY Arg1:E6 Arg2:E5	
T10	Frequency 127 145	2–3 times per week
R3	MODIFY Arg1:T10 Arg2:E5	
T12	Sign_symptom 206 213	dyspnea
E8	Sign_symptom:T12 
T11	Detailed_description 154 180	up to 30 minutes at a time
R4	MODIFY Arg1:T11 Arg2:E5	
T13	Sign_symptom 261 281	regurgitation murmur
E7	Sign_symptom:T13 
T14	Biological_structure 251 260	tricuspid
T15	Detailed_description 238 250	holosystolic
T16	Lab_value 228 237	grade 2/6
R5	MODIFY Arg1:T14 Arg2:E7	
R6	MODIFY Arg1:T15 Arg2:E7	
R7	MODIFY Arg1:T16 Arg2:E7	
T17	Biological_structure 301 320	left sternal border
R8	MODIFY Arg1:T17 Arg2:E

In [10]:
entity_lines = []

for line in annotations.splitlines():
    if line.startswith("T"):
        entity_lines.append(line)

print("Number of entities:", len(entity_lines))
print("\nFirst 10 entites")

for line in entity_lines[:10]:
    print(line)

Number of entities: 68

First 10 entites
T1	Age 8 19	28-year-old
T2	History 20 38	previously healthy
T3	Sex 39 42	man
T4	Clinical_event 43 52	presented
T5	Sign_symptom 31 38	healthy
T6	Duration 60 66	6-week
T7	Sign_symptom 78 90	palpitations
T8	Coreference 96 104	symptoms
T9	Clinical_event 121 125	rest
T10	Frequency 127 145	2–3 times per week


In [13]:
import pandas as pd

entities = []

for line in entity_lines:
    parts = line.split("\t")

    entity_id = parts[0]
    entity_info = parts[1].split()
    entity_text = parts[2]

    entity_type = entity_info[0]
    start = int(entity_info[1])
    end = int(entity_info[2])

    entities.append({
        "entity_id" : entity_id,
        "type" : entity_type,
        "start" : start,
        "end" : end,
        "text" : entity_text
    })

df_entities = pd.DataFrame(entities)

df_entities.head(20)

,entity_id,type,start,end,text
0,T1,Age,8,19,28-year-old
1,T2,History,20,38,previously healthy
2,T3,Sex,39,42,man
3,T4,Clinical_event,43,52,presented
4,T5,Sign_symptom,31,38,healthy
5,T6,Duration,60,66,6-week
6,T7,Sign_symptom,78,90,palpitations
7,T8,Coreference,96,104,symptoms
8,T9,Clinical_event,121,125,rest
9,T10,Frequency,127,145,2–3 times per week


In [14]:
data_dir = Path("../data/MACCROBAT2018")

txt_files = sorted(data_dir.glob("*.txt"))
ann_files = sorted(data_dir.glob("*.ann"))

print("Text files:", len(txt_files))
print("Annotation files:", len(ann_files))

Text files: 200
Annotation files: 200


In [15]:
txt_ids = {file.stem for file in txt_files}
ann_ids = {file.stem for file in ann_files}

print("TXT Ids:", len(txt_ids))
print("ANN Ids:", len(ann_ids))

print("TXT without ANN:", txt_ids - ann_ids)
print("ANN without TXT:", ann_ids - txt_ids)

TXT Ids: 200
ANN Ids: 200
TXT without ANN: set()
ANN without TXT: set()


In [31]:
def parse_ann_file(ann_path):
    entities = []

    annotations = ann_path.read_text(encoding="utf-8")

    for line in annotations.splitlines():
        if line.startswith("T"):
            parts = line.split("\t", 2)

            entity_id = parts[0]
            entity_info = parts[1]
            entity_text = parts[2]

            first_space = entity_info.find(" ")
            entity_type = entity_info[:first_space]
            span_info = entity_info[first_space + 1:]

            spans = []

            for span in span_info.split(";"):
                start, end = span.split()
                spans.append((int(start), int(end)))

            entities.append({
                "entity_id": entity_id,
                "entity_type": entity_type,
                "spans": spans,
                "entity_text": entity_text
            })

    return entities

In [32]:
sample_entities = parse_ann_file(Path("../data/MACCROBAT2018/15939911.ann"))

sample_entities[:5]

[{'entity_id': 'T1',
  'entity_type': 'Age',
  'spans': [(8, 19)],
  'entity_text': '28-year-old'},
 {'entity_id': 'T2',
  'entity_type': 'History',
  'spans': [(20, 38)],
  'entity_text': 'previously healthy'},
 {'entity_id': 'T3',
  'entity_type': 'Sex',
  'spans': [(39, 42)],
  'entity_text': 'man'},
 {'entity_id': 'T4',
  'entity_type': 'Clinical_event',
  'spans': [(43, 52)],
  'entity_text': 'presented'},
 {'entity_id': 'T5',
  'entity_type': 'Sign_symptom',
  'spans': [(31, 38)],
  'entity_text': 'healthy'}]

In [33]:
all_entities = []

for txt_path in txt_files:
    document_id = txt_path.stem

    ann_path = data_dir/ f"{document_id}.ann"

    entities = parse_ann_file(ann_path)

    for entity in entities:
        entity["document_id"] = document_id
        all_entities.append(entity)

df_all_entities = pd.DataFrame(all_entities)

print("Total entities:", len(df_all_entities))

df_all_entities.head()


Total entities: 25041


,entity_id,entity_type,spans,entity_text,document_id
0,T1,Age,"[(8, 19)]",28-year-old,15939911
1,T2,History,"[(20, 38)]",previously healthy,15939911
2,T3,Sex,"[(39, 42)]",man,15939911
3,T4,Clinical_event,"[(43, 52)]",presented,15939911
4,T5,Sign_symptom,"[(31, 38)]",healthy,15939911


In [35]:
df_all_entities["entity_type"].value_counts()

entity_type
Diagnostic_procedure      4567
Sign_symptom              3359
Biological_structure      2931
Detailed_description      2901
Lab_value                 2858
Disease_disorder          1362
Medication                1076
Therapeutic_procedure     1005
Date                       731
Clinical_event             626
History                    392
Severity                   369
Dosage                     362
Nonbiological_location     354
Coreference                313
Duration                   280
Age                        206
Sex                        191
Administration             175
Distance                   122
Activity                   108
Family_history              81
Frequency                   76
Shape                       65
Time                        57
Personal_background         57
Subject                     54
Color                       52
Texture                     46
Area                        43
Outcome                     42
Qualitative_concept        

In [36]:
print("Total_entities:", len(df_all_entities))
print("Entity_types:", df_all_entities["entity_type"].nunique())
print("Documents:", df_all_entities["document_id"].nunique())


Total_entities: 25041
Entity_types: 41
Documents: 200
